In [1]:
import ee
import geemap

ee.Authenticate(auth_mode='colab')
ee.Initialize(project='adv-python-practicals-255601')

# Create an interactive map centered on Mumbai
Map = geemap.Map(center=[19.0, 72.9], zoom=10)

# Define Region of Interest (ROI)
roi = ee.Geometry.Rectangle([72.7, 18.8, 73.0, 19.3])  # [West, South, East, North]

# Add ROI boundary to the map
Map.addLayer(roi, {'color': 'red'}, 'ROI - Mumbai')
Map.centerObject(roi, 10)

# Load Sentinel-2 Surface Reflectance Image Collection
collection = (
    ee.ImageCollection('COPERNICUS/S2_SR')
    .filterBounds(roi)
    .filterDate('2024-01-01', '2024-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
)

# Print number of images used
print("Total images used:", collection.size().getInfo())

# Define NDWI computation function (Green = B3, NIR = B8)
def add_ndwi(image):
    ndwi = image.normalizedDifference(['B3', 'B8']).rename('NDWI')
    return image.addBands(ndwi)

# Apply NDWI function to image collection
ndwi_collection = collection.map(add_ndwi)

# Compute mean NDWI for the entire period
mean_ndwi = ndwi_collection.select('NDWI').mean().clip(roi)

# Visualization parameters
ndwi_vis = {
    'min': -0.5,
    'max': 0.5,
    'palette': ['brown', 'white', 'blue']
}

# Add NDWI layer to map
Map.addLayer(mean_ndwi, ndwi_vis, 'Mean NDWI (Water Bodies)')
Map


Total images used: 160


Map(center=[19.049935018708513, 72.85000000000034], controls=(WidgetControl(options=['position', 'transparent_…